In [1]:
#######################################################################################################################################################################
########################################### This notebook takes the cleaned data file and format it for Google Earth Engine ##########################################
#######################################################################################################################################################################

In [2]:
# Import the necessary packages
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import json

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import ee # Import the Python API package
import geemap # The geemap Python package is built upon the ipyleaflet and folium packages and implements several methods for interacting with Earth Engine data layer

try:
    ee.Initialize(project='amibea') # Initialize the API
except Exception as e:
    ee.Authenticate() # Authenticate to the Earth Engine servers
    ee.Initialize(project='amibea') # Initialize the API

In [3]:
# Import the raw data from a CSV file into a data frame
pathToFile = r'E:\Amibatec\Updated2026\TA_Endemicity.csv' # Change path to your data
rawData = pd.read_csv(pathToFile,sep=";",encoding= 'unicode_escape')
Data = pd.DataFrame(rawData)
print(Data.shape)
print(Data.head(10))

(1300, 2)
    latitude   longitude
0  80.183543   52.853088
1  79.866876   10.298928
2  79.491876   13.032261
3  79.452293   13.284344
4  79.429376   13.280177
5  79.146043  102.723913
6  78.908543   12.076011
7  78.902293   12.113511
8  78.902293   12.105178
9  78.900210   12.105178


In [4]:
# Export the result to disc
Data.to_csv(r'E:\Amibatec\Updated2026\TA_Endemicity_GEEformated.csv', index = False)

In [5]:
# Export an ee.FeatureCollection as an Earth Engine asset.
# create a tmp gdf
import geopandas as gpd
import json

df = pd.read_csv(r'E:\Amibatec\Updated2026\TA_Endemicity_GEEformated.csv', sep=',', engine='python')
gdf = gpd.GeoDataFrame(
    df, 
    crs='EPSG:4326', 
    geometry = gpd.points_from_xy(
        df['longitude'], 
        df['latitude']
    )
)
    
# convert it into geo-json 
json_df = json.loads(gdf.to_json())
    
# create a gee object with geemap
ee_object = geemap.geojson_to_ee(json_df)
            
#create and launch the task
task = ee.batch.Export.table.toAsset(
    collection= ee_object,
    description= 'exportToTableAsset',
    assetId= 'projects/amibea/assets/TA_Endemicity_GEEformated_2026',
)
task.start()